In [5]:
import sys
sys.path.append('../')
import model_loader, state_utils, evaluate as evaluate_module, data, autoencoder, plot, utils
import copy
import yaml
import torch
import pandas as pd
import csv
import time
import numpy as np
NUM_RUNS = 5

In [7]:
print(sys.path)

['E:\\Python\\python311.zip', 'E:\\Python\\DLLs', 'E:\\Python\\Lib', 'E:\\Python', 'd:\\Hu_Module\\Master\\Semester 4\\Study Project\\Linear attention state management\\venv', '', 'd:\\Hu_Module\\Master\\Semester 4\\Study Project\\Linear attention state management\\venv\\Lib\\site-packages', '../']


In [9]:
from pathlib import Path

config_path = Path("configs/config1.yaml")
if not config_path.exists():
    config_path = (Path.cwd() / ".." / "configs" / "config1.yaml").resolve()

config = utils.read_config(config_path)

paths = config["paths"]
output_dir = paths["output_dir"]
text_history_dir = paths["text_history_dir"]+"/history.txt"
state_dir = paths["state_dir"]+"/state.pt"
plot_dir = paths["plot_dir"]
experiment1_path = output_dir + "/experiment1/experiment1.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"

In [13]:
from pathlib import Path

df = pd.read_csv("../results/experiment1/experiment1.csv")
latency_df = df[["turn", "baseline_latency", "state_latency"]]
size_df = df[["turn", "txt_size_kb", "pt_size_kb"]]
Path(plot_dir + "/experiment1").mkdir(parents=True, exist_ok=True)
plot.plot_memory_growth(size_df, plot_dir + "/experiment1/memory_growth.png")
plot.plot_latency_comparison(latency_df, plot_dir + "/experiment1/latency_comparison.png")
plot.plot_ppl_comparison(df[["turn", "baseline_ppl", "state_ppl"]], plot_dir + "/experiment1/perplexity_comparison.png")

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "state-spaces/mamba2-130m"
tokenizer = AutoTokenizer.from_pretrained("state-spaces/mamba-130m-hf")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map="cuda"
)

KeyboardInterrupt: 